In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from pathlib import Path

# -----------------------------
# Create output folders
# -----------------------------
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("reports/charts").mkdir(parents=True, exist_ok=True)

# -----------------------------
# Connect to SQLite
# -----------------------------
engine = create_engine("sqlite:///../data/db/bluestock_mf.db")

# -----------------------------
# Load NAV data
# -----------------------------
query = """
SELECT
    n.amfi_code,
    n.date_id,
    n.nav,
    f.scheme_name,
    f.risk_category
FROM fact_nav n
JOIN dim_fund f
    ON n.amfi_code = f.amfi_code
ORDER BY n.amfi_code, n.date_id
"""

nav = pd.read_sql(query, engine)

# -----------------------------
# Prepare data
# -----------------------------
nav["date_id"] = pd.to_datetime(nav["date_id"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

nav = nav.dropna(subset=["daily_return"])

# -----------------------------
# Calculate VaR & CVaR
# -----------------------------
var_results = []

for code, grp in nav.groupby("amfi_code"):

    r = grp["daily_return"].values

    # Skip funds with insufficient history
    if len(r) < 30:
        continue

    # Historical VaR (95%)
    var_95 = np.percentile(r, 5)

    # Historical CVaR (Expected Shortfall)
    cvar_95 = r[r <= var_95].mean()

    var_results.append({
        "amfi_code": code,
        "scheme_name": grp["scheme_name"].iloc[0],
        "risk_category": grp["risk_category"].iloc[0],
        "var_95_pct": round(var_95 * 100, 4),
        "cvar_95_pct": round(cvar_95 * 100, 4),
        "worst_day_pct": round(r.min() * 100, 4),
        "best_day_pct": round(r.max() * 100, 4),
        "n_trading_days": len(r)
    })

# -----------------------------
# Results dataframe
# -----------------------------
var_df = pd.DataFrame(var_results)

if var_df.empty:
    raise ValueError(
        "No VaR results generated. Check fact_nav and dim_fund tables."
    )

var_df = var_df.sort_values("var_95_pct")

# Save CSV
csv_path = "data/processed/var_cvar_report.csv"
var_df.to_csv(csv_path, index=False)

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 6))

colors_map = {
    "Very High": "#E24B4A",
    "High": "#BA7517",
    "Moderate": "#534AB7",
    "Low": "#1D9E75"
}

for cat in var_df["risk_category"].dropna().unique():

    sub = var_df[var_df["risk_category"] == cat]

    ax.scatter(
        sub["var_95_pct"],
        sub["cvar_95_pct"],
        label=cat,
        color=colors_map.get(cat, "#888780"),
        s=60,
        alpha=0.85
    )

    for _, row in sub.iterrows():
        label = row["scheme_name"][:15]

        ax.annotate(
            label,
            (row["var_95_pct"], row["cvar_95_pct"]),
            fontsize=7,
            alpha=0.6,
            xytext=(3, 3),
            textcoords="offset points"
        )

ax.set_xlabel("VaR 95% (Daily %)")
ax.set_ylabel("CVaR 95% (Daily %)")
ax.set_title(
    "Historical VaR vs CVaR — All Schemes (Colour = Risk Category)"
)

ax.axhline(
    0,
    color="gray",
    linewidth=0.5,
    linestyle="--"
)

ax.legend()

plt.tight_layout()

chart_path = "reports/charts/16_var_cvar_scatter.png"

plt.savefig(
    chart_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

# -----------------------------
# Output
# -----------------------------
print("\nTop 10 Highest-Risk Funds")
print(
    var_df[
        ["scheme_name", "var_95_pct", "cvar_95_pct"]
    ]
    .head(10)
    .to_string(index=False)
)

print(f"\n✅ Saved: {csv_path}")
print(f"✅ Saved: {chart_path}")

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/20/e3q8)